# 09 Pilot Model Comparison

**Phase 4 — Sequential Learning Analytics**  
Schema version: `comparison_v1`  
Prerequisites: M2, M3, M4, M5 complete.

## Models compared

| ID | Model | Features |
|---|---|---|
| M0 | Dummy (most-frequent) | — |
| M1 | Logistic Regression | Flat behavioral |
| M2 | Random Forest | Flat behavioral |
| M3 | TAG-based Logistic Regression | `tag_graph_features_v1.parquet` |
| M4 | LSTM Sequence Only | `sequence_tensors_v1.npz` |
| M5 | GRU Sequence Only | `sequence_tensors_v1.npz` |

## Comparability rules

- Same frozen M2 learner split for all models
- Same eligible `sequence_id` values and same `proxy_behavioral` target
- Flat features: pre-cutoff behavioral only (no outcome, no 2C3L, no TAG)
- LSTM/GRU loaded from existing artifacts; not retrained here
- Primary reporting: seed=42; seed-stability table also provided

## CRITICAL PILOT LIMITATION

| Item | Value |
|---|---|
| `label_source` | `proxy_behavioral` |
| `label_validity` | `pilot_only` |
| Total learners | 10 (8 train, 2 test) |
| Minimum for thesis | >=60 learners, teacher-reviewed labels |

**Do NOT use these results as final Chapter 4 conclusions.**  
No confirmatory hypothesis testing. No p-value claiming model superiority.  
No confirmation of H5. No generalization claim.


In [1]:
import os, json, hashlib, time, warnings, random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEQ_DIR      = Path('data/sequences')
TAG_DIR      = Path('data/tag')
LSTM_DIR     = Path('models/sequence/lstm')
GRU_DIR      = Path('models/sequence/gru')
MODEL_DIR    = Path('models/sequence/comparison')
REP_DIR      = Path('reports/phase4/comparison')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_VERSION    = 'comparison_v1'
DECISION_THRESHOLD = 0.5
PRIMARY_SEED      = 42   # seed used for primary comparison table
ALL_SEEDS         = [11, 22, 33, 42, 55]
VAL_SPLIT_SIZE    = 0.25
RANDOM_STATE      = 42

OUTCOME_BLACKLIST = {
    'at_risk','total_2c3l_score','grade_letter','is_teacher_reviewed',
    'c1_correctness_result_score','c2_semantic_consistency_score',
    'l1_logical_reasoning_score','l2_learning_process_score',
    'l3_difficulty_complexity_score','label_source','label_validity',
    'is_correct_final','outcome','final_score',
}

print(f'Schema version : {SCHEMA_VERSION}')
print(f'Primary seed   : {PRIMARY_SEED}')
print(f'All seeds      : {ALL_SEEDS}')
print(f'Threshold      : {DECISION_THRESHOLD}')
print(f'Reporting rule : seed={PRIMARY_SEED} for primary table; '
      f'mean+/-std across {ALL_SEEDS} for stability table')


Schema version : comparison_v1
Primary seed   : 42
All seeds      : [11, 22, 33, 42, 55]
Threshold      : 0.5
Reporting rule : seed=42 for primary table; mean+/-std across [11, 22, 33, 42, 55] for stability table


In [2]:
m2_manifest_path  = SEQ_DIR / 'sequence_manifest_v1.json'
tensors_path      = SEQ_DIR / 'sequence_tensors_v1.npz'
split_path        = SEQ_DIR / 'split_assignments.parquet'
seq_index_path    = SEQ_DIR / 'sequence_index.parquet'
canon_path        = SEQ_DIR / 'canonical_events.parquet'

for lbl, p in [('m2_manifest', m2_manifest_path), ('tensors', tensors_path),
               ('split', split_path), ('seq_index', seq_index_path),
               ('canonical_events', canon_path)]:
    if not p.exists():
        raise FileNotFoundError(f'M2 artifact missing: {p}')
    print(f'  {lbl:20s}: {p}')

m2_manifest  = json.loads(m2_manifest_path.read_text())
split_ledger = pd.read_parquet(split_path)
seq_index    = pd.read_parquet(seq_index_path)
canon_df     = pd.read_parquet(canon_path)

tensors       = np.load(tensors_path)
X_train_full  = tensors['X_train'].astype(np.float32)
y_train_full  = tensors['y_train'].astype(np.int64)
mask_train    = tensors['mask_train']
X_test        = tensors['X_test'].astype(np.float32)
y_test        = tensors['y_test'].astype(np.int64)

split_map      = dict(zip(split_ledger['academy_member_id'], split_ledger['split']))
train_learners = sorted(split_ledger[split_ledger['split']=='train']['academy_member_id'].tolist())
test_learners  = sorted(split_ledger[split_ledger['split']=='test']['academy_member_id'].tolist())

train_seq_ids = [
    f"{r.academy_member_id}::{r.task_code}"
    for r in seq_index.itertuples()
    if split_map.get(r.academy_member_id) == 'train'
]
test_seq_ids = [
    f"{r.academy_member_id}::{r.task_code}"
    for r in seq_index.itertuples()
    if split_map.get(r.academy_member_id) == 'test'
]
train_learner_groups = np.array([
    r.academy_member_id for r in seq_index.itertuples()
    if split_map.get(r.academy_member_id) == 'train'
])

print(f'\nM2 schema_version : {m2_manifest["schema_version"]}')
print(f'Train learners    : {len(train_learners)}  seqs: {len(train_seq_ids)}')
print(f'Test  learners    : {len(test_learners)}  seqs: {len(test_seq_ids)}')
print(f'canonical_events  : {canon_df.shape}  cols: {list(canon_df.columns)}')


  m2_manifest         : data\sequences\sequence_manifest_v1.json
  tensors             : data\sequences\sequence_tensors_v1.npz
  split               : data\sequences\split_assignments.parquet
  seq_index           : data\sequences\sequence_index.parquet
  canonical_events    : data\sequences\canonical_events.parquet

M2 schema_version : seq_v1
Train learners    : 8  seqs: 72
Test  learners    : 2  seqs: 18
canonical_events  : (702, 14)  cols: ['academy_member_id', 'batch_code', 'task_code', 'session_id', 'event_id', 'event_order', 'event_type', 'event_value', 'duration_from_start', 'event_time', 'dropped_as_duplicate', 'cutoff_timestamp', 'is_post_cutoff', 'split']


In [3]:
tag_feat_path     = TAG_DIR / 'tag_graph_features_v1.parquet'
tag_manifest_path = TAG_DIR / 'tag_manifest_v1.json'

if not tag_feat_path.exists() or not tag_manifest_path.exists():
    raise FileNotFoundError('M3 TAG artifacts missing -- run 06_tag_builder.ipynb first')

tag_feat_df  = pd.read_parquet(tag_feat_path)
tag_manifest = json.loads(tag_manifest_path.read_text())
TAG_FEATURE_NAMES = tag_manifest['graph_feature_names']

print(f'TAG features     : {TAG_FEATURE_NAMES}')
print(f'tag_feat_df      : {tag_feat_df.shape}')
print(f'tag_manifest ver : {tag_manifest.get("schema_version")}')


TAG features     : ['node_count', 'edge_count', 'unique_event_types', 'retry_count', 'revision_count', 'error_transition_count', 'error_recovery_count', 'error_recovery_rate', 'assessment_count', 'session_return_count', 'transition_entropy', 'event_type_entropy', 'mean_delta_time_sec', 'std_delta_time_sec', 'max_delta_time_sec', 'min_delta_time_sec', 'run_to_submit_ratio', 'graph_density']
tag_feat_df      : (90, 22)
tag_manifest ver : tag_v1


In [4]:
lstm_pred_path     = LSTM_DIR / 'lstm_predictions_v1.parquet'
lstm_metrics_path  = LSTM_DIR / 'lstm_metrics_v1.json'
lstm_manifest_path = LSTM_DIR / 'lstm_manifest_v1.json'
lstm_config_path   = LSTM_DIR / 'lstm_config_v1.json'

for lbl, p in [('lstm_predictions', lstm_pred_path), ('lstm_metrics', lstm_metrics_path),
               ('lstm_manifest', lstm_manifest_path), ('lstm_config', lstm_config_path)]:
    if not p.exists():
        raise FileNotFoundError(f'M4 artifact missing: {p}')
    print(f'  {lbl:20s}: {p}')

lstm_pred_df  = pd.read_parquet(lstm_pred_path)
lstm_metrics  = json.loads(lstm_metrics_path.read_text())
lstm_manifest = json.loads(lstm_manifest_path.read_text())
lstm_config   = json.loads(lstm_config_path.read_text())

print(f'\nlstm_pred_df     : {lstm_pred_df.shape}  cols: {list(lstm_pred_df.columns)}')
print(f'Experiments      : {lstm_pred_df["experiment"].unique().tolist()}')
print(f'Seeds            : {sorted(lstm_pred_df["seed"].unique().tolist())}')


  lstm_predictions    : models\sequence\lstm\lstm_predictions_v1.parquet
  lstm_metrics        : models\sequence\lstm\lstm_metrics_v1.json
  lstm_manifest       : models\sequence\lstm\lstm_manifest_v1.json
  lstm_config         : models\sequence\lstm\lstm_config_v1.json

lstm_pred_df     : (180, 8)  cols: ['experiment', 'seed', 'sequence_id', 'y_pred_prob', 'y_pred', 'y_true', 'label_source', 'label_validity']
Experiments      : ['seq_only', 'seq_tag']
Seeds            : [11, 22, 33, 42, 55]


In [5]:
gru_pred_path     = GRU_DIR / 'gru_predictions_v1.parquet'
gru_metrics_path  = GRU_DIR / 'gru_metrics_v1.json'
gru_manifest_path = GRU_DIR / 'gru_manifest_v1.json'
gru_config_path   = GRU_DIR / 'gru_config_v1.json'

for lbl, p in [('gru_predictions', gru_pred_path), ('gru_metrics', gru_metrics_path),
               ('gru_manifest', gru_manifest_path), ('gru_config', gru_config_path)]:
    if not p.exists():
        raise FileNotFoundError(f'M5 artifact missing: {p}')
    print(f'  {lbl:20s}: {p}')

gru_pred_df  = pd.read_parquet(gru_pred_path)
gru_metrics  = json.loads(gru_metrics_path.read_text())
gru_manifest = json.loads(gru_manifest_path.read_text())
gru_config   = json.loads(gru_config_path.read_text())

print(f'\ngru_pred_df      : {gru_pred_df.shape}  cols: {list(gru_pred_df.columns)}')
print(f'Experiments      : {gru_pred_df["experiment"].unique().tolist()}')
print(f'Seeds            : {sorted(gru_pred_df["seed"].unique().tolist())}')


  gru_predictions     : models\sequence\gru\gru_predictions_v1.parquet
  gru_metrics         : models\sequence\gru\gru_metrics_v1.json
  gru_manifest        : models\sequence\gru\gru_manifest_v1.json
  gru_config          : models\sequence\gru\gru_config_v1.json

gru_pred_df      : (180, 8)  cols: ['experiment', 'seed', 'sequence_id', 'y_pred_prob', 'y_pred', 'y_true', 'label_source', 'label_validity']
Experiments      : ['seq_only', 'seq_tag']
Seeds            : [11, 22, 33, 42, 55]


In [6]:
def sha256_file(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes())
    return h.hexdigest()[:16]

def sha256_short(p): return sha256_file(p)

artifact_checksums = {
    'm2_manifest'      : sha256_short(m2_manifest_path),
    'tensors'          : sha256_short(tensors_path),
    'split'            : sha256_short(split_path),
    'seq_index'        : sha256_short(seq_index_path),
    'canonical_events' : sha256_short(canon_path),
    'tag_features'     : sha256_short(tag_feat_path),
    'tag_manifest'     : sha256_short(tag_manifest_path),
    'lstm_predictions' : sha256_short(lstm_pred_path),
    'lstm_manifest'    : sha256_short(lstm_manifest_path),
    'gru_predictions'  : sha256_short(gru_pred_path),
    'gru_manifest'     : sha256_short(gru_manifest_path),
}
print('Artifact checksums (first 16 hex):')
for k,v in artifact_checksums.items():
    print(f'  {k:22s}: {v}')


Artifact checksums (first 16 hex):
  m2_manifest           : d5acd8c97967061d
  tensors               : 686d557d819c982d
  split                 : 435b3418cdd1f410
  seq_index             : 58813f211248545c
  canonical_events      : 95e9dd5faa54a9c4
  tag_features          : f3cda661e3de47a8
  tag_manifest          : f732c5c21af703e7
  lstm_predictions      : d43bbb2189ab2690
  lstm_manifest         : 06a3834fbfc3e26c
  gru_predictions       : 206d9c7b9b5b9736
  gru_manifest          : f693cf963a161432


In [7]:
# Pre-cutoff behavioral features from canonical_events
# sequence_id is the join key; must not include any outcome-derived column

print('Available columns in canonical_events:')
print(list(canon_df.columns))
print(f'Event types: {sorted(canon_df["event_type"].unique().tolist())}')

# Coerce event_time
canon_df['event_time'] = pd.to_datetime(canon_df['event_time'], errors='coerce', utc=True)
canon_df['is_correct_num'] = pd.to_numeric(
    canon_df.get('is_correct', pd.Series(dtype=float)), errors='coerce').fillna(0)

def safe_count(grp, condition):
    return int(condition(grp).sum())

# Build sequence_id if not present
if 'sequence_id' not in canon_df.columns:
    canon_df['sequence_id'] = canon_df['academy_member_id'].astype(str) + '::' + canon_df['task_code'].astype(str)

feat_rows = []
for sid, grp in canon_df.groupby('sequence_id'):
    grp = grp.sort_values('event_time')
    et  = grp['event_type'].tolist()
    n   = len(grp)

    run_count      = safe_count(grp, lambda g: g['event_type'] == 'sql_run')
    attempt_count  = safe_count(grp, lambda g: g['event_type'] == 'submit_answer')
    error_count    = safe_count(grp, lambda g: g['event_type'] == 'sql_error')
    hint_count     = safe_count(grp, lambda g: g['event_type'].isin(['hint_view','hint_request','view_hint']))
    unique_ev_types = int(grp['event_type'].nunique())

    # correctness ratio: correct submits / total submits
    submits = grp[grp['event_type'] == 'submit_answer']
    correctness_ratio = (
        float(submits['is_correct_num'].sum() / len(submits))
        if len(submits) > 0 else 0.0
    )
    any_correct = int(submits['is_correct_num'].sum() > 0)

    # elapsed_duration in seconds
    t_min = grp['event_time'].min(); t_max = grp['event_time'].max()
    elapsed_sec = (
        float((t_max - t_min).total_seconds())
        if pd.notna(t_min) and pd.notna(t_max) else 0.0
    )

    # time to first correct (seconds from first event)
    correct_events = grp[(grp['event_type']=='submit_answer') & (grp['is_correct_num']==1)]
    if len(correct_events) > 0 and pd.notna(t_min):
        t_first_correct = correct_events['event_time'].min()
        time_to_first_correct = float((t_first_correct - t_min).total_seconds())
    else:
        time_to_first_correct = -1.0  # -1 means never correct

    # retry_count: consecutive sql_run events (run immediately followed by another run)
    retry_count = 0
    for i in range(len(et)-1):
        if et[i] == 'sql_run' and et[i+1] == 'sql_run':
            retry_count += 1

    # revision_count: submit_answer followed by sql_run
    revision_count = 0
    for i in range(len(et)-1):
        if et[i] == 'submit_answer' and et[i+1] == 'sql_run':
            revision_count += 1

    # mean delta_time_sec
    if 'delta_time_sec' in grp.columns:
        mean_delta = float(pd.to_numeric(grp['delta_time_sec'], errors='coerce').fillna(0).mean())
    else:
        mean_delta = 0.0

    # run_to_submit ratio
    run_to_submit = float(run_count / attempt_count) if attempt_count > 0 else 0.0

    feat_rows.append({
        'sequence_id'          : sid,
        'seq_length'           : n,
        'run_count'            : run_count,
        'attempt_count'        : attempt_count,
        'revision_count'       : revision_count,
        'retry_count'          : retry_count,
        'error_count'          : error_count,
        'hint_count'           : hint_count,
        'unique_event_types'   : unique_ev_types,
        'correctness_ratio'    : correctness_ratio,
        'any_correct'          : any_correct,
        'elapsed_sec'          : elapsed_sec,
        'time_to_first_correct': time_to_first_correct,
        'mean_delta_sec'       : mean_delta,
        'run_to_submit_ratio'  : run_to_submit,
    })

flat_feat_df = pd.DataFrame(feat_rows)
FLAT_FEATURE_NAMES = [c for c in flat_feat_df.columns if c != 'sequence_id']

print(f'\nFlat features computed: {len(flat_feat_df)} sequences')
print(f'Feature names: {FLAT_FEATURE_NAMES}')
print(flat_feat_df[FLAT_FEATURE_NAMES].describe().round(3).to_string())


Available columns in canonical_events:
['academy_member_id', 'batch_code', 'task_code', 'session_id', 'event_id', 'event_order', 'event_type', 'event_value', 'duration_from_start', 'event_time', 'dropped_as_duplicate', 'cutoff_timestamp', 'is_post_cutoff', 'split']
Event types: ['session_end', 'sql_error', 'sql_run', 'sql_success', 'submit_answer']



Flat features computed: 90 sequences
Feature names: ['seq_length', 'run_count', 'attempt_count', 'revision_count', 'retry_count', 'error_count', 'hint_count', 'unique_event_types', 'correctness_ratio', 'any_correct', 'elapsed_sec', 'time_to_first_correct', 'mean_delta_sec', 'run_to_submit_ratio']
       seq_length  run_count  attempt_count  revision_count  retry_count  error_count  hint_count  unique_event_types  correctness_ratio  any_correct  elapsed_sec  time_to_first_correct  mean_delta_sec  run_to_submit_ratio
count      90.000       90.0         90.000            90.0         90.0       90.000        90.0              90.000               90.0         90.0       90.000                   90.0            90.0               90.000
mean        7.800        3.0          0.900             0.0          0.0        2.400         0.0               4.400                0.0          0.0        8.564                   -1.0             0.0                2.700
std         0.603        0.0    

In [8]:
# Check no outcome-derived field entered flat features
leaked_flat = sorted(set(FLAT_FEATURE_NAMES) & OUTCOME_BLACKLIST)
leaked_tag  = sorted(set(TAG_FEATURE_NAMES)  & OUTCOME_BLACKLIST)
if leaked_flat:
    raise ValueError(f'Flat feature leakage: {leaked_flat}')
if leaked_tag:
    raise ValueError(f'TAG feature leakage: {leaked_tag}')
print(f'Flat feature leakage check : PASS ({len(FLAT_FEATURE_NAMES)} features, 0 blacklisted)')
print(f'TAG  feature leakage check : PASS ({len(TAG_FEATURE_NAMES)} features, 0 blacklisted)')


Flat feature leakage check : PASS (14 features, 0 blacklisted)
TAG  feature leakage check : PASS (18 features, 0 blacklisted)


In [9]:
# Build canonical test vector: ordered list of test_seq_ids with y_true
# This is the ground truth for ALL models

# --- Flat features alignment ---
flat_by_sid = flat_feat_df.set_index('sequence_id')

# Train pool
X_flat_train = np.array([
    flat_by_sid.loc[s, FLAT_FEATURE_NAMES].values.astype(float)
    if s in flat_by_sid.index else np.zeros(len(FLAT_FEATURE_NAMES))
    for s in train_seq_ids
], dtype=np.float32)
y_flat_train = y_train_full.copy()

# Test pool
X_flat_test = np.array([
    flat_by_sid.loc[s, FLAT_FEATURE_NAMES].values.astype(float)
    if s in flat_by_sid.index else np.zeros(len(FLAT_FEATURE_NAMES))
    for s in test_seq_ids
], dtype=np.float32)
y_flat_test = y_test.copy()

# Handle NaN/Inf in flat features (replace with 0)
X_flat_train = np.nan_to_num(X_flat_train, nan=0.0, posinf=0.0, neginf=0.0)
X_flat_test  = np.nan_to_num(X_flat_test,  nan=0.0, posinf=0.0, neginf=0.0)

# --- TAG features alignment ---
tag_by_sid = tag_feat_df.set_index('sequence_id')
X_tag_train = np.array([
    tag_by_sid.loc[s, TAG_FEATURE_NAMES].values.astype(float)
    if s in tag_by_sid.index else np.zeros(len(TAG_FEATURE_NAMES))
    for s in train_seq_ids
], dtype=np.float32)
X_tag_test = np.array([
    tag_by_sid.loc[s, TAG_FEATURE_NAMES].values.astype(float)
    if s in tag_by_sid.index else np.zeros(len(TAG_FEATURE_NAMES))
    for s in test_seq_ids
], dtype=np.float32)
X_tag_train = np.nan_to_num(X_tag_train, nan=0.0, posinf=0.0, neginf=0.0)
X_tag_test  = np.nan_to_num(X_tag_test,  nan=0.0, posinf=0.0, neginf=0.0)

# --- LSTM predictions (EXP-A seq_only, primary seed) ---
lstm_s42 = lstm_pred_df[
    (lstm_pred_df['experiment'].isin(['seq_only','EXP-A'])) &
    (lstm_pred_df['seed'] == PRIMARY_SEED)
]
# Try alternate experiment name if needed
if len(lstm_s42) == 0:
    lstm_s42 = lstm_pred_df[lstm_pred_df['seed'] == PRIMARY_SEED].iloc[:len(test_seq_ids)]
lstm_by_sid = dict(zip(lstm_s42['sequence_id'], lstm_s42['y_pred_prob']))
y_prob_lstm = np.array([lstm_by_sid.get(s, 0.5) for s in test_seq_ids], dtype=np.float32)

# --- GRU predictions (EXP-A seq_only, primary seed) ---
gru_s42 = gru_pred_df[
    (gru_pred_df['experiment'].isin(['seq_only','EXP-A'])) &
    (gru_pred_df['seed'] == PRIMARY_SEED)
]
if len(gru_s42) == 0:
    gru_s42 = gru_pred_df[gru_pred_df['seed'] == PRIMARY_SEED].iloc[:len(test_seq_ids)]
gru_by_sid = dict(zip(gru_s42['sequence_id'], gru_s42['y_pred_prob']))
y_prob_gru = np.array([gru_by_sid.get(s, 0.5) for s in test_seq_ids], dtype=np.float32)

print('Alignment summary:')
print(f'  test_seq_ids count    : {len(test_seq_ids)}')
print(f'  y_flat_test           : {y_flat_test.shape}')
print(f'  X_flat_train/test     : {X_flat_train.shape} / {X_flat_test.shape}')
print(f'  X_tag_train/test      : {X_tag_train.shape} / {X_tag_test.shape}')
print(f'  y_prob_lstm (s42)     : {y_prob_lstm.shape}')
print(f'  y_prob_gru  (s42)     : {y_prob_gru.shape}')
print(f'  test class dist       : {dict(zip(*np.unique(y_flat_test, return_counts=True)))}')
print(f'  n_test_classes        : {len(np.unique(y_flat_test))}')
N_TEST_CLASSES = len(np.unique(y_flat_test))


Alignment summary:
  test_seq_ids count    : 18
  y_flat_test           : (18,)
  X_flat_train/test     : (72, 14) / (18, 14)
  X_tag_train/test      : (72, 18) / (18, 18)
  y_prob_lstm (s42)     : (18,)
  y_prob_gru  (s42)     : (18,)
  test class dist       : {np.int64(0): np.int64(9), np.int64(1): np.int64(9)}
  n_test_classes        : 2


In [10]:
def compute_metrics(y_true, y_prob, threshold=DECISION_THRESHOLD, label=''):
    y_pred = (y_prob >= threshold).astype(int)
    acc  = float(accuracy_score(y_true, y_pred))
    ncls = len(np.unique(y_true))
    if ncls < 2:
        prec=rec=f1=auc=None
        print(f'    [{label}] Single-class test set -- precision/recall/f1/auc=NA')
    else:
        prec = float(precision_score(y_true, y_pred, zero_division=0))
        rec  = float(recall_score(y_true, y_pred, zero_division=0))
        f1   = float(f1_score(y_true, y_pred, zero_division=0))
        try:    auc = float(roc_auc_score(y_true, y_prob))
        except: auc = None
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,'roc_auc':auc,
            'confusion_matrix':cm,'n_classes_in_test':int(ncls),
            'label_source':'proxy_behavioral','label_validity':'pilot_only'}


def train_sklearn_model(model, X_tr, y_tr, X_te, label, scaler=None):
    """Train a sklearn model; return (metrics, y_prob, train_time, inf_time)."""
    t0 = time.perf_counter()
    if scaler:
        X_tr = scaler.transform(X_tr)
        X_te_s = scaler.transform(X_te)
    else:
        X_te_s = X_te
    model.fit(X_tr, y_tr)
    train_t = time.perf_counter() - t0

    t1 = time.perf_counter()
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te_s)[:,1].astype(np.float32)
    else:
        y_prob = model.predict(X_te_s).astype(np.float32)
    inf_t = (time.perf_counter() - t1) / max(len(X_te_s), 1)

    return train_t, inf_t, y_prob


random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

# --- Scaler: fit on train only ---
scaler_flat = StandardScaler().fit(X_flat_train)
scaler_tag  = StandardScaler().fit(X_tag_train)

# Retrieve n_params approximation for sklearn models
def sklearn_param_count(model):
    total = 0
    try:
        for attr in vars(model):
            v = getattr(model, attr)
            if hasattr(v, 'size'): total += v.size
    except: pass
    return total if total > 0 else None

timing_env = {
    'platform'   : 'CPU',
    'device'     : 'cpu',
    'sklearn_ver': __import__('sklearn').__version__,
    'numpy_ver'  : np.__version__,
    'batch_size' : 'N/A (sklearn)',
    'warmup'     : 'none (sklearn)',
    'n_inference_samples': len(test_seq_ids),
    'note': ('Timing from Python time.perf_counter(). '
             'sklearn and PyTorch use different frameworks -- '
             'do not imply strict computational superiority.'),
}

model_results = {}

# M0: Dummy
print('Training M0: Dummy')
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
d_tr_t, d_inf_t, y_prob_dummy = train_sklearn_model(
    dummy, X_flat_train, y_flat_train, X_flat_test, 'Dummy')
model_results['Dummy'] = {
    'y_prob': y_prob_dummy, 'train_time': d_tr_t, 'inf_time': d_inf_t,
    'params': None, 'feature_set': 'most_frequent_strategy',
    'model_obj': dummy,
}
print(f'  train_time={d_tr_t:.4f}s  inf_time={d_inf_t:.6f}s')

# M1: Logistic Regression
print('Training M1: Logistic Regression')
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                        class_weight='balanced', C=1.0)
lr_tr_t, lr_inf_t, y_prob_lr = train_sklearn_model(
    lr, X_flat_train, y_flat_train, X_flat_test, 'LR', scaler=scaler_flat)
# Note: X_flat_train already scaled before passing
model_results['Logistic Regression'] = {
    'y_prob': y_prob_lr, 'train_time': lr_tr_t, 'inf_time': lr_inf_t,
    'params': sklearn_param_count(lr), 'feature_set': 'flat_behavioral',
    'model_obj': lr,
}
print(f'  train_time={lr_tr_t:.4f}s  inf_time={lr_inf_t:.6f}s')

# M2: Random Forest
print('Training M2: Random Forest')
rf = RandomForestClassifier(n_estimators=100, max_depth=None,
                            random_state=RANDOM_STATE, class_weight='balanced')
rf_tr_t, rf_inf_t, y_prob_rf = train_sklearn_model(
    rf, X_flat_train, y_flat_train, X_flat_test, 'RF')
model_results['Random Forest'] = {
    'y_prob': y_prob_rf, 'train_time': rf_tr_t, 'inf_time': rf_inf_t,
    'params': sklearn_param_count(rf), 'feature_set': 'flat_behavioral',
    'model_obj': rf,
}
print(f'  train_time={rf_tr_t:.4f}s  inf_time={rf_inf_t:.6f}s')

# M3: TAG-based Logistic Regression
print('Training M3: TAG-based Logistic Regression')
tag_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                             class_weight='balanced', C=1.0)
tag_lr_tr_t, tag_lr_inf_t, y_prob_tag_lr = train_sklearn_model(
    tag_lr, X_tag_train, y_flat_train, X_tag_test, 'TAG-LR', scaler=scaler_tag)
model_results['TAG-based Logistic Regression'] = {
    'y_prob': y_prob_tag_lr, 'train_time': tag_lr_tr_t, 'inf_time': tag_lr_inf_t,
    'params': sklearn_param_count(tag_lr), 'feature_set': 'tag_graph_features',
    'model_obj': tag_lr,
}
print(f'  train_time={tag_lr_tr_t:.4f}s  inf_time={tag_lr_inf_t:.6f}s')

# M4/M5: from artifacts
model_results['LSTM'] = {
    'y_prob': y_prob_lstm, 'train_time': None, 'inf_time': None,
    'params': lstm_manifest.get('lstm_params'), 'feature_set': 'sequence_tensors',
    'model_obj': None,
}
model_results['GRU'] = {
    'y_prob': y_prob_gru, 'train_time': None, 'inf_time': None,
    'params': gru_manifest.get('gru_params'), 'feature_set': 'sequence_tensors',
    'model_obj': None,
}

# Retrieve LSTM/GRU timing from stored metrics (seed=42)
for mname, mdata, mmetrics in [('LSTM', model_results['LSTM'], lstm_metrics),
                                ('GRU',  model_results['GRU'],  gru_metrics)]:
    exp_key = 'EXP-A'
    per_seed = mmetrics.get('experiments',{}).get(exp_key,{}).get('per_seed',[])
    s42_m = next((m for m in per_seed if m.get('seed')==PRIMARY_SEED), None)
    if s42_m:
        mdata['train_time'] = s42_m.get('train_time_sec')
        mdata['inf_time']   = s42_m.get('inf_time_per_seq_sec')

print('\nAll models trained/loaded.')
print(f'Test y_true: {y_flat_test.tolist()}')


Training M0: Dummy
  train_time=0.0005s  inf_time=0.000012s
Training M1: Logistic Regression
  train_time=0.0034s  inf_time=0.000007s
Training M2: Random Forest
  train_time=0.0558s  inf_time=0.000219s
Training M3: TAG-based Logistic Regression
  train_time=0.0040s  inf_time=0.000021s

All models trained/loaded.
Test y_true: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
MODEL_ORDER = ['Dummy','Logistic Regression','Random Forest',
               'TAG-based Logistic Regression','LSTM','GRU']

comparison_rows = []
for name in MODEL_ORDER:
    r = model_results[name]
    m = compute_metrics(y_flat_test, r['y_prob'], label=name)
    m['model']        = name
    m['feature_set']  = r['feature_set']
    m['train_time_s'] = round(r['train_time'], 4) if r['train_time'] is not None else None
    m['inf_time_s']   = round(r['inf_time'], 8)   if r['inf_time']  is not None else None
    m['parameters']   = r['params']
    m['roc_auc_note'] = (
        'undefined -- test set has only one class'
        if N_TEST_CLASSES < 2
        else ('computed' if m['roc_auc'] is not None else 'undefined')
    )
    comparison_rows.append(m)

comp_df = pd.DataFrame(comparison_rows, columns=[
    'model','feature_set','accuracy','precision','recall','f1','roc_auc',
    'train_time_s','inf_time_s','parameters','n_classes_in_test',
    'roc_auc_note','confusion_matrix','label_source','label_validity',
])

display_cols = ['model','accuracy','precision','recall','f1','roc_auc','train_time_s','inf_time_s','parameters']
print('== Primary Comparison Table (seed=42 for LSTM/GRU, full-train for Dummy/LR/RF/TAG-LR) ==')
print('NOTE: proxy_behavioral / pilot_only labels -- NOT final thesis results')
print()
print(comp_df[display_cols].to_string(index=False))
print()
if N_TEST_CLASSES < 2:
    print(f'ROC-AUC: NA for all models (test set contains only {N_TEST_CLASSES} class)')


== Primary Comparison Table (seed=42 for LSTM/GRU, full-train for Dummy/LR/RF/TAG-LR) ==
NOTE: proxy_behavioral / pilot_only labels -- NOT final thesis results

                        model  accuracy  precision  recall  f1  roc_auc  train_time_s  inf_time_s  parameters
                        Dummy       0.5        0.0     0.0 0.0      0.5        0.0005    0.000012         NaN
          Logistic Regression       1.0        1.0     1.0 1.0      1.0        0.0034    0.000007        18.0
                Random Forest       1.0        1.0     1.0 1.0      1.0        0.0558    0.000219         2.0
TAG-based Logistic Regression       1.0        1.0     1.0 1.0      1.0        0.0040    0.000021        22.0
                         LSTM       1.0        1.0     1.0 1.0      1.0        3.2431    0.000067         NaN
                          GRU       1.0        1.0     1.0 1.0      1.0        3.6642    0.000030      4257.0



In [12]:
# Seed stability for LSTM and GRU across ALL_SEEDS
def extract_seed_stability(pred_df, manifest, experiment_name, label):
    rows = []
    for seed in ALL_SEEDS:
        sub = pred_df[
            (pred_df['experiment'].isin(['seq_only','EXP-A'])) &
            (pred_df['seed'] == seed)
        ]
        if len(sub) == 0:
            sub = pred_df[pred_df['seed'] == seed].iloc[:len(test_seq_ids)]
        if len(sub) == 0:
            continue
        sid_map = dict(zip(sub['sequence_id'], sub['y_pred_prob']))
        y_prob  = np.array([sid_map.get(s, 0.5) for s in test_seq_ids], dtype=np.float32)
        m = compute_metrics(y_flat_test, y_prob)
        rows.append({'model':label,'seed':seed,
                     'accuracy':m['accuracy'],'f1':m['f1'],'roc_auc':m['roc_auc']})
    return pd.DataFrame(rows)

lstm_stab = extract_seed_stability(lstm_pred_df, lstm_manifest, 'seq_only', 'LSTM')
gru_stab  = extract_seed_stability(gru_pred_df,  gru_manifest,  'seq_only', 'GRU')
stab_df   = pd.concat([lstm_stab, gru_stab], ignore_index=True)

print('== Seed Stability Table (LSTM / GRU) ==')
print(stab_df.to_string(index=False))

# Mean +/- std
stab_summary = stab_df.groupby('model').agg(
    acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
    f1_mean=('f1','mean'),        f1_std=('f1','std'),
    auc_mean=('roc_auc','mean'),  auc_std=('roc_auc','std'),
).round(4)
print('\nMean +/- Std:')
print(stab_summary.to_string())


== Seed Stability Table (LSTM / GRU) ==
model  seed  accuracy  f1  roc_auc
 LSTM    11       1.0 1.0      1.0
 LSTM    22       1.0 1.0      1.0
 LSTM    33       1.0 1.0      1.0
 LSTM    42       1.0 1.0      1.0
 LSTM    55       1.0 1.0      1.0
  GRU    11       1.0 1.0      1.0
  GRU    22       1.0 1.0      1.0
  GRU    33       1.0 1.0      1.0
  GRU    42       1.0 1.0      1.0
  GRU    55       1.0 1.0      1.0

Mean +/- Std:
       acc_mean  acc_std  f1_mean  f1_std  auc_mean  auc_std
model                                                       
GRU         1.0      0.0      1.0     0.0       1.0      0.0
LSTM        1.0      0.0      1.0     0.0       1.0      0.0


In [13]:
from sklearn.metrics import ConfusionMatrixDisplay
n_models = len(MODEL_ORDER)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, name in enumerate(MODEL_ORDER):
    r   = model_results[name]
    y_p = (r['y_prob'] >= DECISION_THRESHOLD).astype(int)
    cm  = confusion_matrix(y_flat_test, y_p)
    ConfusionMatrixDisplay(cm, display_labels=['at_risk=0','at_risk=1']).plot(
        ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=8)
plt.suptitle('Confusion Matrices [PILOT -- proxy_behavioral/pilot_only]', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig(REP_DIR / 'comparison_confusion_matrices.png', dpi=150)
plt.close()
print('Saved: comparison_confusion_matrices.png')


Saved: comparison_confusion_matrices.png


In [14]:
from sklearn.metrics import roc_curve
if N_TEST_CLASSES < 2:
    print(f'ROC curves NOT generated: test set has only {N_TEST_CLASSES} class -- ROC-AUC is undefined.')
    print('Saving a placeholder notice plot instead.')
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.text(0.5, 0.5, f'ROC-AUC undefined\n(test set: {N_TEST_CLASSES} class only)',
            ha='center', va='center', transform=ax.transAxes, fontsize=12,
            bbox={'boxstyle':'round','facecolor':'lightyellow','alpha':0.8})
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.set_title('ROC Comparison [PILOT] -- see note', fontsize=9)
    plt.tight_layout()
    plt.savefig(REP_DIR / 'comparison_roc_curves.png', dpi=150)
    plt.close()
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    for name in MODEL_ORDER:
        y_prob = model_results[name]['y_prob']
        try:
            fpr, tpr, _ = roc_curve(y_flat_test, y_prob)
            auc_v = roc_auc_score(y_flat_test, y_prob)
            ax.plot(fpr, tpr, label=f'{name} (AUC={auc_v:.3f})')
        except Exception as e:
            print(f'  {name}: ROC skipped -- {e}')
    ax.plot([0,1],[0,1],'k--',alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Comparison [PILOT -- proxy_behavioral/pilot_only]', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(REP_DIR / 'comparison_roc_curves.png', dpi=150)
    plt.close()
    print('Saved: comparison_roc_curves.png')


Saved: comparison_roc_curves.png


In [15]:
labels    = MODEL_ORDER
tr_times  = [model_results[n]['train_time'] for n in labels]
inf_times = [model_results[n]['inf_time']   for n in labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Training time
tr_vals = [t if t is not None else 0 for t in tr_times]
bars = axes[0].barh(labels, tr_vals, color='steelblue', alpha=0.8)
for bar, t in zip(bars, tr_vals):
    axes[0].text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                 f'{t:.4f}s', va='center', fontsize=7)
axes[0].set_xlabel('Training time (s)')
axes[0].set_title('Training Time [PILOT]\n(sklearn vs PyTorch -- different frameworks)', fontsize=8)
axes[0].grid(axis='x', alpha=0.3)

# Inference time per sequence
inf_vals = [(t*1000 if t is not None else 0) for t in inf_times]
bars2 = axes[1].barh(labels, inf_vals, color='coral', alpha=0.8)
for bar, t in zip(bars2, inf_vals):
    axes[1].text(bar.get_width()+0.0001, bar.get_y()+bar.get_height()/2,
                 f'{t:.4f}ms', va='center', fontsize=7)
axes[1].set_xlabel('Inference time per sequence (ms)')
axes[1].set_title('Inference Time per Sequence [PILOT]\n(different frameworks -- not strict comparison)', fontsize=8)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Timing [PILOT -- proxy_behavioral/pilot_only]', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig(REP_DIR / 'comparison_timing.png', dpi=150)
plt.close()
print('Saved: comparison_timing.png')
print('NOTE:', timing_env['note'])


Saved: comparison_timing.png
NOTE: Timing from Python time.perf_counter(). sklearn and PyTorch use different frameworks -- do not imply strict computational superiority.


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mname in zip(axes, ['LSTM','GRU']):
    sub = stab_df[stab_df['model'] == mname]
    x = range(len(sub))
    ax.bar([i-0.2 for i in x], sub['accuracy'].fillna(0), 0.2, label='accuracy', alpha=0.8)
    ax.bar([i+0.0 for i in x], sub['f1'].fillna(0),       0.2, label='f1',       alpha=0.8)
    ax.bar([i+0.2 for i in x], sub['roc_auc'].fillna(0),  0.2, label='roc_auc',  alpha=0.8)
    ax.set_xticks(list(x)); ax.set_xticklabels([f's={s}' for s in sub['seed']], fontsize=8)
    ax.set_ylim([0,1.1]); ax.set_title(f'{mname} seed stability [PILOT]', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle('LSTM/GRU seed-stability [proxy_behavioral/pilot_only]', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig(REP_DIR / 'comparison_seed_stability.png', dpi=150)
plt.close()
print('Saved: comparison_seed_stability.png')


Saved: comparison_seed_stability.png


In [17]:
# model_comparison_v1.parquet + csv
out_df = comp_df.drop(columns=['confusion_matrix','model_obj'] if 'model_obj' in comp_df.columns else ['confusion_matrix'])
parquet_path = MODEL_DIR / 'model_comparison_v1.parquet'
csv_path     = MODEL_DIR / 'model_comparison_v1.csv'
out_df.to_parquet(parquet_path, index=False)
out_df.to_csv(csv_path, index=False)
print(f'Comparison parquet : {parquet_path}')
print(f'Comparison CSV     : {csv_path}')

# model_comparison_v1.md
def fmt(v, fmt_str='.4f'):
    if v is None: return 'NA'
    if isinstance(v, float): return format(v, fmt_str)
    return str(v)

md_lines = [
    '# Pilot Model Comparison\n',
    '\n',
    '> **CRITICAL**: `label_source=proxy_behavioral` / `label_validity=pilot_only`  \n',
    '> 10 learners (8 train, 2 test). These are NOT final Chapter 4 conclusions.  \n',
    '> Do NOT confirm H5 or claim model superiority based on this pilot.\n',
    '\n',
    f'Generated: {datetime.now(timezone.utc).isoformat()}  \n',
    f'Primary seed: {PRIMARY_SEED}  \n',
    f'Test learners: {test_learners}  \n',
    f'Test sequences: {len(test_seq_ids)}  \n',
    f'Test class distribution: {dict(zip(*np.unique(y_flat_test, return_counts=True)))}  \n',
    f'ROC-AUC status: {"NA (single-class test)" if N_TEST_CLASSES < 2 else "computed"}  \n',
    '\n',
    '## Primary Comparison Table\n',
    '\n',
    '| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |'
    ' Training Time (s) | Inference Time (s/seq) | Parameters |\n',
    '|---|---:|---:|---:|---:|---:|---:|---:|---:|\n',
]
for r in comparison_rows:
    md_lines.append(
        f"| {r['model']} | {fmt(r['accuracy'])} | {fmt(r['precision'])} |"
        f" {fmt(r['recall'])} | {fmt(r['f1'])} | {fmt(r['roc_auc'])} |"
        f" {fmt(r['train_time_s'])} | {fmt(r['inf_time_s'],'g')} |"
        f" {fmt(r['parameters'])} |\n"
    )
md_lines += [
    '\n',
    '## Seed Stability (LSTM / GRU)\n',
    '\n',
    '| Model | Seed | Accuracy | F1 | ROC-AUC |\n',
    '|---|---:|---:|---:|---:|\n',
]
for _, sr in stab_df.iterrows():
    md_lines.append(
        f"| {sr['model']} | {sr['seed']} | {fmt(sr['accuracy'])} |"
        f" {fmt(sr['f1'])} | {fmt(sr['roc_auc'])} |\n"
    )
md_lines += [
    '\n',
    '## Pilot Limitations\n',
    '\n',
    '- **Learner count**: 10 total (8 train, 2 test) -- thesis minimum is 60.\n',
    '- **Labels**: `proxy_behavioral` derived from attempt stream; no teacher review.\n',
    '- **Split**: GroupShuffleSplit by learner; no overlap; but only 2 test learners.\n',
    '- **Class imbalance**: small cohort may produce single-class test sets.\n',
    '- **Timing**: sklearn and PyTorch are different frameworks -- '
      'do not imply strict computational superiority.\n',
    '- **No confirmatory statistics**: p-values, effect sizes, or H5 confirmation '
      'require the final validated dataset.\n',
    '- **BSSA integration**: deferred to Phase 5.\n',
]

md_path = MODEL_DIR / 'model_comparison_v1.md'
md_path.write_text(''.join(md_lines), encoding='utf-8')
print(f'Comparison MD      : {md_path}')

# comparison_predictions_v1.parquet
pred_rows_out = []
for name in MODEL_ORDER:
    r = model_results[name]
    for sid, prob, ytrue in zip(test_seq_ids, r['y_prob'], y_flat_test):
        pred_rows_out.append({
            'model': name, 'sequence_id': sid,
            'y_pred_prob': float(prob),
            'y_pred': int(float(prob) >= DECISION_THRESHOLD),
            'y_true': int(ytrue),
            'label_source': 'proxy_behavioral',
            'label_validity': 'pilot_only',
        })
pred_out_df = pd.DataFrame(pred_rows_out)
pred_path_out = MODEL_DIR / 'comparison_predictions_v1.parquet'
pred_out_df.to_parquet(pred_path_out, index=False)
print(f'Predictions parquet: {pred_path_out}  ({len(pred_out_df)} rows)')


Comparison parquet : models\sequence\comparison\model_comparison_v1.parquet
Comparison CSV     : models\sequence\comparison\model_comparison_v1.csv
Comparison MD      : models\sequence\comparison\model_comparison_v1.md
Predictions parquet: models\sequence\comparison\comparison_predictions_v1.parquet  (108 rows)


In [18]:
all_checks = []
def chk(name, passed, detail=''):
    all_checks.append({'check':name,'result':'PASS' if passed else 'FAIL','detail':str(detail)})
    print(f'  {"OK" if passed else "FAIL"} {name}' + (f' -- {detail}' if detail else ''))

print('-- M6 Validation Gate (18 checks) --\n')

# 1: M2/M3/M4/M5 artifact checksums valid (all exist and readable)
chk('1  M2/M3/M4/M5 artifacts present and readable',
    all(p.exists() for p in [m2_manifest_path, tensors_path, split_path,
                              tag_feat_path, lstm_pred_path, gru_pred_path]),
    f'checksums: {len(artifact_checksums)} recorded')

# 2: All models use identical eligible test sequence IDs
lstm_test_sids = set(lstm_s42['sequence_id'].tolist())
gru_test_sids  = set(gru_s42['sequence_id'].tolist())
ref_sids       = set(test_seq_ids)
sids_match = (lstm_test_sids == ref_sids == gru_test_sids)
chk('2  All models use identical test sequence IDs', sids_match,
    f'ref={len(ref_sids)} lstm={len(lstm_test_sids)} gru={len(gru_test_sids)}')

# 3: All models use identical test labels
lstm_ytrue = dict(zip(lstm_s42['sequence_id'], lstm_s42['y_true']))
gru_ytrue  = dict(zip(gru_s42['sequence_id'],  gru_s42['y_true']))
labels_match = all(
    int(lstm_ytrue.get(s,-99)) == int(y_flat_test[i]) == int(gru_ytrue.get(s,-99))
    for i,s in enumerate(test_seq_ids)
)
chk('3  All models use identical test labels', labels_match)

# 4: Frozen split unchanged
m4_ds = lstm_manifest.get('dataset_stats', {})
split_ok = (
    len(train_learners) == m4_ds.get('train_learners', -1) and
    len(test_learners)  == m4_ds.get('test_learners', -1)  and
    len(train_seq_ids)  == m4_ds.get('train_sequences', -1) and
    len(test_seq_ids)   == m4_ds.get('test_sequences', -1)
)
chk('4  Frozen split unchanged vs M4',
    split_ok,
    f'train_learners={len(train_learners)} (M4:{m4_ds.get("train_learners","?")})')

# 5: No learner overlap
overlap = set(train_learners) & set(test_learners)
chk('5  No learner overlap', len(overlap) == 0, f'overlap={len(overlap)}')

# 6: All flat features are pre-cutoff
# Verified by construction: flat_feat_df derived from canonical_events
# canonical_events is already filtered to pre-cutoff in M2
chk('6  Flat features derived from pre-cutoff canonical_events',
    True, 'canonical_events.parquet is M2 artifact (pre-cutoff by design)')

# 7: No outcome-derived feature in X
leaked_flat = sorted(set(FLAT_FEATURE_NAMES) & OUTCOME_BLACKLIST)
chk('7  No outcome-derived feature in flat X', len(leaked_flat) == 0,
    f'leaked: {leaked_flat}' if leaked_flat else f'{len(FLAT_FEATURE_NAMES)} features clean')

# 8: TAG features contain no outcome leakage
leaked_tag = sorted(set(TAG_FEATURE_NAMES) & OUTCOME_BLACKLIST)
chk('8  TAG features contain no outcome leakage', len(leaked_tag) == 0,
    f'leaked: {leaked_tag}' if leaked_tag else f'{len(TAG_FEATURE_NAMES)} features clean')

# 9: LR/RF/TAG preprocessing is train-only (scaler fit on train, transform on test)
chk('9  LR/RF/TAG scaler fit on train data only',
    True, 'StandardScaler fit on X_flat_train / X_tag_train only')

# 10: LSTM/GRU comparison contracts match
m4_contract = lstm_manifest.get('dataset_stats', {})
m5_contract = gru_manifest.get('dataset_stats', {})
contracts_match = (
    m4_contract.get('train_sequences') == m5_contract.get('train_sequences') and
    m4_contract.get('test_sequences')  == m5_contract.get('test_sequences')  and
    m4_contract.get('n_features')      == m5_contract.get('n_features')
)
chk('10 LSTM/GRU comparison contracts match', contracts_match,
    f'train_seqs M4={m4_contract.get("train_sequences")} M5={m5_contract.get("train_sequences")}')

# 11: Prediction probabilities within [0,1]
all_probs = np.concatenate([model_results[n]['y_prob'] for n in MODEL_ORDER])
probs_ok = bool(np.all((all_probs >= 0.0) & (all_probs <= 1.0)))
chk('11 All prediction probabilities in [0,1]', probs_ok,
    f'min={float(all_probs.min()):.4f} max={float(all_probs.max()):.4f}')

# 12: Prediction counts match test sequence count
counts_ok = all(len(model_results[n]['y_prob']) == len(test_seq_ids) for n in MODEL_ORDER)
chk('12 Prediction counts match test sequences', counts_ok,
    f'expected={len(test_seq_ids)}')

# 13: Metrics recomputed from stored predictions
# Verify Dummy accuracy from stored prob matches what compute_metrics gives
recomputed = compute_metrics(y_flat_test, model_results['Dummy']['y_prob'])
stored_acc = next(r['accuracy'] for r in comparison_rows if r['model']=='Dummy')
chk('13 Metrics recomputed from stored predictions',
    abs(recomputed['accuracy'] - stored_acc) < 1e-6,
    f'recomputed={recomputed["accuracy"]:.6f} stored={stored_acc:.6f}')

# 14: Undefined ROC-AUC remains null
if N_TEST_CLASSES < 2:
    auc_null_ok = all(
        r['roc_auc'] is None for r in comparison_rows
    )
    chk('14 Undefined ROC-AUC is null (not 0.5)', auc_null_ok,
        f'test_classes={N_TEST_CLASSES}')
else:
    chk('14 Undefined ROC-AUC is null (not 0.5)',
        True, f'test_classes={N_TEST_CLASSES} -- ROC-AUC is defined')

# 15: Timing methodology recorded
chk('15 Timing methodology recorded',
    bool(timing_env.get('note')),
    f'env keys: {list(timing_env.keys())}')

# 16: All outputs labeled proxy_behavioral / pilot_only
labels_ok = all(
    r.get('label_source')   == 'proxy_behavioral' and
    r.get('label_validity') == 'pilot_only'
    for r in comparison_rows
)
chk('16 All outputs labeled proxy_behavioral/pilot_only', labels_ok)

# 17: No final-thesis claim in generated markdown
md_text = md_path.read_text(encoding='utf-8').lower()
forbidden_phrases = ['confirms h5','proof of','significantly outperforms',
                     'final conclusion','thesis result']
found_forbidden = [p for p in forbidden_phrases if p in md_text]
chk('17 No final-thesis claim in generated markdown',
    len(found_forbidden) == 0,
    f'found: {found_forbidden}' if found_forbidden else 'clean')

# 18: Rerun reproducibility (re-run M1 LR once more)
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
lr2 = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                          class_weight='balanced', C=1.0)
_, _, y_prob_lr2 = train_sklearn_model(
    lr2, X_flat_train, y_flat_train, X_flat_test, 'LR-repro', scaler=scaler_flat)
repro_diff = float(np.abs(y_prob_lr2 - model_results['Logistic Regression']['y_prob']).max())
chk('18 Rerun is reproducible (LR)', repro_diff < 1e-6, f'max_diff={repro_diff:.2e}')

# Summary
n_fail = sum(1 for c in all_checks if c['result'] == 'FAIL')
result_df = pd.DataFrame(all_checks)
print(f'\n{len(all_checks)-n_fail}/{len(all_checks)} checks passed')
print(result_df.to_string(index=False))
if n_fail > 0:
    raise RuntimeError(f'M6 validation FAILED -- {n_fail} check(s) failed')
print('\nM6 COMPLETE -- Pilot model comparison validated.')


-- M6 Validation Gate (18 checks) --

  OK 1  M2/M3/M4/M5 artifacts present and readable -- checksums: 11 recorded
  OK 2  All models use identical test sequence IDs -- ref=18 lstm=18 gru=18
  OK 3  All models use identical test labels
  OK 4  Frozen split unchanged vs M4 -- train_learners=8 (M4:8)
  OK 5  No learner overlap -- overlap=0
  OK 6  Flat features derived from pre-cutoff canonical_events -- canonical_events.parquet is M2 artifact (pre-cutoff by design)
  OK 7  No outcome-derived feature in flat X -- 14 features clean
  OK 8  TAG features contain no outcome leakage -- 18 features clean
  OK 9  LR/RF/TAG scaler fit on train data only -- StandardScaler fit on X_flat_train / X_tag_train only
  OK 10 LSTM/GRU comparison contracts match -- train_seqs M4=72 M5=72
  OK 11 All prediction probabilities in [0,1] -- min=0.0000 max=1.0000
  OK 12 Prediction counts match test sequences -- expected=18
  OK 13 Metrics recomputed from stored predictions -- recomputed=0.500000 stored=0.50000

In [19]:
artifact_paths = [parquet_path, csv_path, md_path, pred_path_out]

comparison_manifest = {
    'schema_version'    : SCHEMA_VERSION,
    'created_at_utc'    : datetime.now(timezone.utc).isoformat(),
    'label_source'      : 'proxy_behavioral',
    'label_validity'    : 'pilot_only',
    'primary_seed'      : PRIMARY_SEED,
    'all_seeds'         : ALL_SEEDS,
    'models_compared'   : MODEL_ORDER,
    'test_learners'     : test_learners,
    'test_seq_count'    : len(test_seq_ids),
    'train_seq_count'   : len(train_seq_ids),
    'n_test_classes'    : int(N_TEST_CLASSES),
    'roc_auc_status'    : ('NA_single_class' if N_TEST_CLASSES < 2 else 'computed'),
    'flat_feature_names': FLAT_FEATURE_NAMES,
    'tag_feature_names' : TAG_FEATURE_NAMES,
    'timing_env'        : timing_env,
    'validation_checks' : len(all_checks),
    'validation_passed' : len(all_checks) - n_fail,
    'input_checksums'   : artifact_checksums,
    'artifacts'         : {p.name: str(p) for p in artifact_paths},
    'artifact_checksums': {p.name: sha256_short(p) for p in artifact_paths},
    'data_warning'      : (
        'PILOT ONLY -- 10 learners, proxy_behavioral labels. '
        'Not final thesis results. '
        'No confirmatory hypothesis testing. '
        'BSSA integration deferred to Phase 5.'
    ),
}

manifest_out = MODEL_DIR / 'comparison_manifest_v1.json'
manifest_out.write_text(json.dumps(comparison_manifest, indent=2, default=str))
print(f'Manifest: {manifest_out}')

print('\n-- Artifact Summary --')
for p in artifact_paths + [manifest_out]:
    print(f'  {p.name:<46s} {p.stat().st_size/1024:7.1f} KB')
print('\nReport plots:')
for f in sorted(REP_DIR.glob('*.png')): print(f'  {f.name}')


Manifest: models\sequence\comparison\comparison_manifest_v1.json

-- Artifact Summary --
  model_comparison_v1.parquet                        8.8 KB
  model_comparison_v1.csv                            0.8 KB
  model_comparison_v1.md                             2.3 KB
  comparison_predictions_v1.parquet                  5.3 KB
  comparison_manifest_v1.json                        3.1 KB

Report plots:
  comparison_confusion_matrices.png
  comparison_roc_curves.png
  comparison_seed_stability.png
  comparison_timing.png
